In [1]:
# Import Libraries

import os
import platform
import random
import shutil
import sys

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import scipy
import sklearn.metrics
from tqdm import tqdm_notebook as tqdm
%matplotlib inline
random.seed(0)

/usr/local/lib/python3.6/dist-packages/statsmodels/tools/_testing.py:19: FutureWarning: pandas.util.testing is deprecated. Use the functions in the public API at pandas.testing instead.
  import pandas.util.testing as tm


In [0]:

from google.colab import drive
drive.mount('/content/gdrive')

Go to this URL in a browser: https://accounts.google.com/o/oauth2/auth?client_id=947318989803-6bn6qk8qdgf4n4g3pfee6491hc0brc4i.apps.googleusercontent.com&redirect_uri=urn%3aietf%3awg%3aoauth%3a2.0%3aoob&response_type=code&scope=email%20https%3a%2f%2fwww.googleapis.com%2fauth%2fdocs.test%20https%3a%2f%2fwww.googleapis.com%2fauth%2fdrive%20https%3a%2f%2fwww.googleapis.com%2fauth%2fdrive.photos.readonly%20https%3a%2f%2fwww.googleapis.com%2fauth%2fpeopleapi.readonly

Enter your authorization code:
··········
Mounted at /content/gdrive


In [0]:
# Import data
from collections import Counter
DIR = '/content/gdrive/My Drive/HLA-E_sequences_shared/'
tsne_comb = pd.DataFrame()
alphabet = 'ACDEFGHIKLMNPQRSTVWY'
alphabet = list(str.split(alphabet))
for R in ['A3', 'C3', 'AA4', 'AC4', 'CA4', 'CC4']:
    data = pd.read_csv(DIR+'3312'+R+'.csv')
    data['Round'] = int(R[-1])
    data['Label'] = R
    print(data.shape)
    data10 = data.loc[data['count']>10]
    print(data10.shape)
    tsne_comb =tsne_comb.append(data, ignore_index = True)
print(tsne_comb)

(12119, 13)
(1124, 13)
(10563, 13)
(1671, 13)
(4729, 13)
(902, 13)
(5713, 13)
(954, 13)
(6170, 13)
(704, 13)
(5952, 13)
(689, 13)
        sequence  count  0  1  2  3  4  5  6  7  8  Round Label
0      AAADPCWKC      1  A  A  A  D  P  C  W  K  C      3    A3
1      AAAEAPLTS      1  A  A  A  E  A  P  L  T  S      3    A3
2      AAERMDPAF      2  A  A  E  R  M  D  P  A  F      3    A3
3      AAGHPVTEI      3  A  A  G  H  P  V  T  E  I      3    A3
4      AAGIFYSRT      1  A  A  G  I  F  Y  S  R  T      3    A3
...          ...    ... .. .. .. .. .. .. .. .. ..    ...   ...
45241  YYIPIGEWG      3  Y  Y  I  P  I  G  E  W  G      4   CC4
45242  YYPSRSLFL      1  Y  Y  P  S  R  S  L  F  L      4   CC4
45243  YYPSRSLLW    289  Y  Y  P  S  R  S  L  L  W      4   CC4
45244  YYPSRSLWL      2  Y  Y  P  S  R  S  L  W  L      4   CC4
45245  YYPYRSLLW      1  Y  Y  P  Y  R  S  L  L  W      4   CC4

[45246 rows x 13 columns]


In [0]:
# For each library in round 3 or 4, count aa frequencies at each position
for R in ['R0','A1','A2','A3','AA4','AC4','C1','C2','C3','CA4','CC4']:
    data = pd.read_csv(DIR+'3312'+R+'.csv')
    data['Round'] = int(R[-1])
    data['Label'] = R
    if R in ['A3', 'C3', 'AA4', 'AC4', 'CA4', 'CC4']:
      print(R)
      counts_list = []
      for ii in ['0', '1', '2', '3', '4', '5', '6', '7', '8']:
        counts = Counter(data[ii])
        counts_list.append(counts)
        #print(counts) # This prints the positional aa frequencies
      # counts_list contains 9 dictionaries, one for positional freq at each position

A3
AA4
AC4
C3
CA4
CC4


In [0]:
# Run Chi-Squared test on each pair of aa positions, cramers_v statistics
A3 = tsne_comb.loc[tsne_comb['Label']=='A3']
chisqr_dict = {}
cramer_vs = {}
def cramers_stat(chi2, covar_freqs):
    #chi2 = scipy.stats.chi2_contingency(confusion_matrix)[0]
    #n = confusion_matrix.sum()
    covar_np = covar_freqs.to_numpy()
    n = sum(sum(covar_np))
    covar_probs = covar_np/n
    return (np.sqrt(chi2 / (n*(min(covar_freqs.shape)-1))), covar_probs)

for ii in range(9):
  for i2 in range(9):
      if i2>ii:
        var1 = str(ii)
        var2 = str(i2)
        covar_freqs = pd.crosstab(A3[var1],A3[var2],margins = False)
        chisqr, p, df, exp  = scipy.stats.chi2_contingency(covar_freqs)
        cramer_v, covar_probs = cramers_stat(chisqr, covar_freqs)
        chisqr_r = round(chisqr, 1)
        cramer_v = round(cramer_v, 3)
        covar_probs = pd.DataFrame(data=covar_probs)
        #print(covar_probs)
        chisqr_dict[var1+var2] = (chisqr_r, p, cramer_v)
print(chisqr_dict)

In [0]:
# The sections below focus on cross-selection data
# AC refers to A library selected with NKG2C
AA4 = tsne_comb.loc[tsne_comb['Label']=='AA4']
AC4 = tsne_comb.loc[tsne_comb['Label']=='AC4']

In [0]:
# Calculate overlap between AA4 and AC4, 
# get counts of overlap peptides peptides in AA4 and AC4 respectively
AA_overlap = AA4[AA4["sequence"].isin(AC4["sequence"])]
#A_overlap['Count_AC'] = AC.loc[AC['Count'] for AC["sequence"]==A_overlap["sequence"]]
AC_overlap = AC4[AC4["sequence"].isin(AA4["sequence"])]
A_overlap = pd.merge(AA_overlap, AC_overlap, on='sequence')
print(A_overlap.shape)
print(A_overlap['count_x'])
print(A_overlap['count_y'])
#A_overlap10 = A_overlap.loc[A_overlap['count']>10]
#print(A_overlap10.shape)

(1663, 25)
0        12
1        19
2        17
3         1
4         2
       ... 
1658    947
1659      6
1660      1
1661      1
1662     12
Name: count_x, Length: 1663, dtype: int64
0        280
1          1
2         49
3          1
4         54
        ... 
1658    2528
1659       8
1660       3
1661       1
1662       2
Name: count_y, Length: 1663, dtype: int64


In [0]:
CC4 = tsne_comb.loc[tsne_comb['Label']=='CC4']
CA4 = tsne_comb.loc[tsne_comb['Label']=='CA4']

In [0]:
# C_overlap: peptides that overlap between CC4 and CA4
# C_overlap10: peptides that overlap between CC4 and CA4, with CC4 count>10
C_overlap = CC4[CC4["sequence"].isin(CA4["sequence"])]
print(C_overlap.shape)
#print(C_overlap)
C_overlap10 = C_overlap.loc[C_overlap['count']>10]
print(C_overlap10.shape)
CA4_overlap10 = CA4[CA4["sequence"].isin(C_overlap10["sequence"])]
counts = CA4_overlap10["count"].to_numpy()
C_overlap10["count_CA4"] = counts
#with pd.option_context('display.max_rows', None, 'display.max_columns', None):  # more options can be specified also
#    print(C_overlap10)

In [0]:
# C_not_overlap: peptides in CC4 with counts>10, where don't show up in CA4
C_10 = CC4.loc[CC4["count"]>10]
print(C_10.shape)
C_not_overlap = C_10[~C_10["sequence"].isin(C_overlap["sequence"])]
print(C_not_overlap.shape)
print(C_not_overlap)

(689, 13)
(117, 13)
        sequence  count  0  1  2  3  4  5  6  7  8  Round Label
39327  AFVMWPRFS     17  A  F  V  M  W  P  R  F  S      4   CC4
39329  AGPSRSVFL     40  A  G  P  S  R  S  V  F  L      4   CC4
39357  AKARPFQII     23  A  K  A  R  P  F  Q  I  I      4   CC4
39401  ALPHRTLLM     91  A  L  P  H  R  T  L  L  M      4   CC4
39466  AMPDRSLFW     14  A  M  P  D  R  S  L  F  W      4   CC4
...          ...    ... .. .. .. .. .. .. .. .. ..    ...   ...
45057  WWELGTKII    136  W  W  E  L  G  T  K  I  I      4   CC4
45116  YGWDIKQIF     11  Y  G  W  D  I  K  Q  I  F      4   CC4
45162  YLPMFLPKS     12  Y  L  P  M  F  L  P  K  S      4   CC4
45194  YSHDSVLIK     13  Y  S  H  D  S  V  L  I  K      4   CC4
45225  YVPDRSLWL     17  Y  V  P  D  R  S  L  W  L      4   CC4

[117 rows x 13 columns]


In [0]:
# Identify peptides present in C3 what are in C_not_overlap
C3 = tsne_comb.loc[tsne_comb['Label']=='C3']
print(C3.shape)
C3_not_overlap = C3.loc[C3["sequence"].isin(C_not_overlap["sequence"])]
print(C3_not_overlap)

(10563, 13)
        sequence  count  0  1  2  3  4  5  6  7  8  Round Label
12198  AFVMWPRFS     28  A  F  V  M  W  P  R  F  S      3    C3
12215  AGPSRSVFL     93  A  G  P  S  R  S  V  F  L      3    C3
12260  AKARPFQII     74  A  K  A  R  P  F  Q  I  I      3    C3
12356  ALPHRTLLM    241  A  L  P  H  R  T  L  L  M      3    C3
12493  AMPDRSLFW     43  A  M  P  D  R  S  L  F  W      3    C3
...          ...    ... .. .. .. .. .. .. .. .. ..    ...   ...
22182  WNRLLDFRE    110  W  N  R  L  L  D  F  R  E      3    C3
22186  WNRLLMSKK      9  W  N  R  L  L  M  S  K  K      3    C3
22198  WNRVLPNAY      1  W  N  R  V  L  P  N  A  Y      3    C3
22308  WWELGTKII    390  W  W  E  L  G  T  K  I  I      3    C3
22499  YLPMFLPKS      3  Y  L  P  M  F  L  P  K  S      3    C3

[66 rows x 13 columns]


In [0]:
# Making list of A3 sequences, with or without duplicates
A3 = tsne_comb.loc[tsne_comb['Label']=='A3']
A3 = A3.loc[A3['count']>100]
# A3 duplicates for seq logo, round(count/20)
A3_sequences = A3["sequence"].tolist()
total_seq = []
for index, row in A3.iterrows():
  total_seq.extend([row["sequence"]]*int(np.round(row["count"]/130)))
print(len(total_seq))
total_seq = '\n'.join(total_seq)
A3_sequences = ' '.join(A3_sequences)
#print(A3_sequences)

9341
AAPVRSLFL AGRFGRLLS AIPSRTRWL AKARPFQII ALGGRTLWL ALPGRTVWL ALPWRSVWL ALTGRALWL AMAPRTLLL AMTQRSLWL ARLYPWRIW ASWFDFIYS ATPARSLFL ATPDRTLWL AVPERSRFL CAPAFMFYN CFNRLLVVS CGPAWIFLR CVNERAHGT CWNRIIVLK CWNRLLNPE DCLITDTVS DCMIWAPAA DFPLGFLTR DSFGQTWVI ELTIPWIWS EMSWLAWLY ESRLSRLDI EVFPLGFLP FAPYRSLIF FGGRIIRGS FGGRLIHHR FGGRLLPYR FGGRLLSNI FLPERSLIF FLPFRSVWM FLPSRSIMF FMGHRSLFL FMGHRSLWL FMPHRSLWM FQSHRALFL FRASLSVNS FRPELMRLM FRWHDLWYL FSGVYPPWM FWFPFLRPE FYFPSGFLV FYPWRSLWL GAPAWMFYD GELMGFMRY GELMGMLRY GGFFGRIID GGFWGRLLA GGKYGRLLM GGMFGRLLG GGRFGRLLS GGRWGRLLL GIPDRSLFL GLNGRTMFL GPTGLGIEQ GRKRMLWPG GRSLSRLYT GSSWLNYLY GYLPPWLPI GYPSRSLFF GYWGQILKM HFPERSVFM HRQAPSYRS HVVWAANEL IAPSRSVFL IAPYRSRFL IEGWYPPFL IGKYGRILD IGPLRSLLF IGSHRTPYL IKDNWAPFF IKDNWVPFF IKPRRSLFF ILADRSLWL ILPHRSLWV ILPNRWVWL IMAHRSLML IMSARTLWL IPPGRTLLF IQALRTLWL IRADRTMWL IRADRTVVF IRPVRTLWL ISAGRTVVF ISAHRSRWL ISPHRSIFF ITPERTLWL IVHDLFPFY IVPGRSLLF IYPERSLFL KEALAPVPI KIPLRSVFM KLPGRTLIF KLPNRSLWI KSLWG

In [0]:
# C3 duplicates for seq logo
def make_duplicates(data, count_ratio):
  """ Input dataframe, for peptides count>10, make duplicates based on round(counts/count_ratio)
  Return sequences in peptide form (1 per line)"""
  #data = tsne_comb.loc[tsne_comb['Label']==r]
  data = data.loc[data['count']>10]
  total_seq = []
  for index, row in data.iterrows():
    total_seq.extend([row["sequence"]]*int(np.round(row["count"]/count_ratio)))
  print(len(total_seq))
  total_seq = '\n'.join(total_seq)
  return total_seq

In [0]:
DIR = '/content/gdrive/My Drive/HLA-E_sequences_shared/'
def save_text(seqs, name):
  """ Saves seqs in file specified by DIR and name"""
  if os.path.exists(DIR+name):
    os.remove(DIR+name)
  text_file = open(DIR+name, "w")
  n = text_file.write(seqs)
  text_file.close()

In [0]:
# Save sequences without duplicates in peptide form
def save_sequences(data, name, threshold = 10, comb10 = False):
  """ Save sequences without duplicates in peptides form
  Takes in a dataframe, gets unique peptides with counts above threshold
  Saves sequences in peptide form (1 line/pep) under file name provided """
#  if comb10: 
#    data = tsne_comb10.loc[tsne_comb10['Label']==r]
#  else:
#    data = tsne_comb.loc[tsne_comb['Label']==r]
  data = data.loc[data['count']>threshold]
  print(data.shape)
  seq = data['sequence']
  total_seq = '\n'.join(seq)
  save_text(total_seq, name)
  return seq

In [0]:
# Load 10mer data, and combine A3, A4, C3 into one dataframe
DIR2 = '/content/gdrive/My Drive/HLA-E 10mer data/'
tsne_comb10 = pd.DataFrame()
# A3 data problematic, Use A4, C3
for R in ['A3', 'C3', 'A4']:
    data = pd.read_csv(DIR2+'10mer_'+R+'.csv')
    data['Round'] = int(R[-1])
    data['Label'] = R
    print(data.shape)
    data10 = data.loc[data['count']>10]
    print(data10.shape)
    tsne_comb10 =tsne_comb10.append(data, ignore_index = True)

(1935, 14)
(27, 14)
(6225, 14)
(510, 14)
(7909, 14)
(1294, 14)


In [0]:
#Set different restrictions to identify 10mer C3 peptides alternative motifs
C3 = tsne_comb10.loc[tsne_comb10['Label']=='C3']
C3 = C3.loc[C3['count']>10]
print(C3.shape)
# Remove peptides that don't have P5R or P9L 
C3_other = C3[(C3['4'] != 'R') | ((C3['8'] != 'L') & (C3['8'] != 'F') & (C3['8'] != 'M') & (C3['8'] != 'W'))]
C3_89 = C3[((C3['8'] != 'L') & (C3['8'] != 'F') & (C3['8'] != 'M') & (C3['8'] != 'W')) & 
              ((C3['9'] == 'L') | (C3['9'] == 'F') | (C3['9'] == 'M') | (C3['9'] == 'W'))]
C3_strict = C3[(C3['4'] != 'R') & (C3['8'] != 'L')]
#pd.set_option('display.max_rows', None)
print(C3_other.shape)
#C3_other
#C3_other.to_csv(DIR+'C3_alt.csv', index=False)
print(C3_strict.shape)
#C3_strict.to_csv(DIR+'C3_10mer_strict.csv', index=False)
#print(C3_89.shape)
#C3_89


(510, 14)
(118, 14)
(63, 14)


In [0]:
# Code from here below are all function calls or saving outputs 
C3_89.to_csv(DIR+'C3_10mer_910.csv', index=False)

In [0]:
save_text(make_duplicates(C3_other, 30), 'C3_alt')

4842


In [0]:
save_text(make_duplicates(C3_strict, 30), 'C3_10mer_strict')

1094


In [0]:
save_sequences(C3_other, 'C3_alt_1')
save_sequences(C3_strict, 'C3_strict')

(118, 14)
(63, 14)


1952    ADVASKMGRN
1960    AGARISESSG
2468    CLLEKARWVL
2519    DFSSFARPEN
2546    DPRIHSYPEV
           ...    
7964    WNRLLPLRRA
8111    YNRIIKLKPA
8135    YSMPLTLSRF
8136    YSQLHYYGNP
8155    YVSVCHIAQS
Name: sequence, Length: 63, dtype: object

In [0]:
C3 = save_sequences('C3', 'C3_seq', threshold = 10)
C3_10 = save_sequences('C3', 'C3_10mer_seqs', comb10 = True)

(1671, 13)
(510, 14)


In [0]:
# Saved merged C3 sequences text file
C3 = C3.append(C3_10)
print(len(C3))
C3_seqs = '\n'.join(C3)
save_text(C3_seqs, 'C3_merged_seqs')

2181


In [0]:
total_seqAA4 = make_duplicates('AA4', 150)
save_text(total_seqAA4, 'AA4_seq_150')

9698


In [0]:
total_seqCC4 = make_duplicates('CC4', 150)
save_text(total_seqCC4, 'CC4_seq_150')

7964


In [0]:
total_seqCA4 = make_duplicates('CA4', 150)
save_text(total_seqCA4, 'CA4_seq_150')

9332


In [0]:
total_seqAC4 = make_duplicates('AC4', 150)
save_text(total_seqAC4, 'AC4_seq_150')

9568


In [0]:
save_text(make_duplicates('C3', 150), 'C3_seq_150')

9053


In [0]:
save_text(make_duplicates('A3', 150), 'A3_seq_150')

8099
